# Basic Hado API Usage
This notebook will go through the basic usage of the hado API which allows for a scriptable interface for the design automation of hollowframe DNA origami nanostructures. Overall, the Streamlit GUI (online, found [here](hado-origami.streamlit.app/hado)) adds overhead and is a bit slower to execute the package (but is obviously more accessible). I would recommend using the scripting interface if you are at-all comfortable with Python as it will be a smoother experience.

The notebook will install the package when you run the next cell (presuming you haven't installed hado yet)

Overall, the purpose of this notebook is to show how the data is transformed from a list of vertices, edges, and a specified number of helices-per-edge into the scaffold / staple sequences required for the manufacturing of a hollowframe nanostructure. The API can also be used in-the-loop with generative design tools (i.e., `mango` found here [here](https://github.com/CMU-Integrated-Design-Innovation-Group/Mango), enabling users to customize their own search spaces. 

In [ ]:
%%capture
!git clone https://github.com/ajvetturini/hado.git
!pip install -e hado

In [1]:
from hado import HadoManager, StapleArgs, ScaffoldArgs, Geometry
from pathlib import Path

In [2]:
# You can programatically defined vertices as 2D list of (x, y, z) poitns and the edges as a 2D list
# connecting [vi, vj] and i cannot equal j and must be bound by the length of vertices
# n_per_edge is the number of DNA helices per edge that you'd want 
geometry = Geometry(
    vertices=[[0., 0., 0.], [35., 0., 0.], [-35., 0., 0.], [0., 35., 0.], [0., -35., 0.]],  # MUST BE FLOAT!!!
    edges=[[0, 1], [0, 2], [0, 3], [0, 4]], 
    n_per_edge=18
)

# Alternatively, you can read in a mesh file
# geometry = Geometry.read_in_mesh(path/to/mesh, n_per_edge=6)

In [3]:
# The ScaffoldArgs controls your scaffold sequence (to use when writing output staple sequences). 
# You can read the full args (likely do not need to be changed) in the readthedocs
scaf_args = ScaffoldArgs(scaffold_sequence='m13')  # Other options are 'p7308', 'p7560', 'p7704', 'p8064', 'p8100', 'p8634'

# You could also specify a custom sequence:
custom_seq = 'ACGTA'  # Can only contain A C G and T, short seq here as this is just an example
custom_scaf_args = ScaffoldArgs(scaffold_sequence=custom_seq)

In [4]:
# The StapleArgs contain a variety of parameters that you can read in the docs, but 
# the more common options are shown below:
staple_args = StapleArgs(
    only_add=False,  # False is default option, if set True then no staple breaks are placed resulting in very long staples
    min_run_post_xover=3,  # Number of nts a staple must bind to scaffold before a nick can be placed
)

In [ ]:
# Finally, you can create a manager with these objects. Note that if you use the defaults (which is M13) you actually
# only need to specify a geometry. i.e., scaf_args = ScaffoldArgs() and stap_args = StapleArgs() are valid 
manager = HadoManager('Plus_Sign', geometry, scaf_args, staple_args)

# Alternatively you could read in a *.hado file (i.e., from the UI)
# This file contains the geometry, ScaffoldArgs, and StapleArgs (covered in next two cells)
## manager = HadoManager.load('docs/plus_sign.hado')

In [ ]:
# You can simply run the design automation via the run command below
# This returns a PipelineDiagnostics object that contains metadata about the design automation process
diagnostics = manager.run()

In [ ]:
# Finally, output to your favorite CAD / CAE tool (if you need another format supported, please let me know!)
# filename_no_extension can be set in all write function but by default uses the design name (Plus_Sign)
manager.write_sequences(filename_no_extension='My_Sequences.csv')  
manager.write_cadnano()
manager.write_scadnano(filename_no_extension='MyCustomFilename')

# oxDNA can have the old topology format set (default is to use old top)
manager.write_oxDNA(use_old_top=True) 
manager.write_oxView() 